# Emu3.5 Vision Tokenizer + Apertus VLM Demo

This notebook demonstrates:
1. **Image Reconstruction** with the Emu3.5 IBQ vision tokenizer
2. **Image Captioning** using the Apertus VLM
3. **Visual Question Answering** — provide an image + prompt and get a response

**Requirements:**
- Emu3.5 vision tokenizer checkpoint
- Apertus model checkpoint (HF format, SFT/instruct variant)
- Text tokenizer (with EMU3 special tokens)

**First-time setup:**

1. **Initialize git submodules** (includes the Emu3.5 tokenizer code):
```bash
git submodule update --init --recursive
```

2. **Apply the Emu3.5 spatial indices patch** — this modifies Emu3.5 to return
   spatial indices `[B, H, W]` instead of flattened indices, which is required
   by the `Emu3_5_IBQ` tokenizer wrapper. From the repository root:
```bash
cd Tokenizer/submodules/Emu3.5
git apply ../../patches/emu35_spatial_indices.patch
```

See `Tokenizer/patches/README.md` for details.

## 0. Setup

We need specififc transformers version for apertus. Install other missing packages as needed.

In [ ]:
%pip install omegaconf
%pip install --upgrade "transformers>=4.56.0,<5.0"

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import os
import pathlib
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import gc

# Add project root to path
PROJECT_ROOT = pathlib.Path(os.getcwd()).parent.parent
sys.path.append(str(PROJECT_ROOT))
sys.path.append(str(PROJECT_ROOT / "Tokenizer"))
print(f"Project root: {PROJECT_ROOT}")

# GPU info
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        free, total = torch.cuda.mem_get_info(i)
        print(f"  GPU {i} ({props.name}): {free/1024**3:.1f}GB free / {total/1024**3:.1f}GB total")

In [ ]:
# ── Paths (edit these to match your setup) ──

# Emu3.5 vision tokenizer checkpoint (contains config.yaml + model.ckpt)
VISION_TOKENIZER_PATH = "/capstor/store/cscs/swissai/infra01/MLLM/tokenizer/Emu3.5-VisionTokenizer"

# Text tokenizer (with EMU3 special tokens + vision tokens)
TEXT_TOKENIZER_PATH = "/capstor/store/cscs/swissai/infra01/MLLM/tokenizer/apertus_emu3.5_instruct"

# Apertus model checkpoint (HF checkpoint)
# NOTE: The default paths point to an instruct/SFT checkpoint. If you use a
#       base (non-instruct) checkpoint, you also need to update the InferenceArgs
#       in Section 2 (set apply_chat_template=False, chat_transform=None).
APERTUS_MODEL_PATH = "/capstor/store/cscs/swissai/infra01/MLLM/ablations/apertus-8b-img-SFT-32nodes-gbs512-mbs1-steps8030-img-text-seqlen8192-s2onlytxtloss/HF"  # <-- EDIT THIS

# Device configuration
VISION_DEVICE = "cuda:0"   # GPU for vision tokenizer
MODEL_DEVICE = "cuda:0"    # GPU for Apertus LLM (use different GPU if available)

# Resolution bounds for smart resize (images larger of smaller will be resized to this max/min number of pixels for tokenization)
MIN_PIXELS = 256 * 256
MAX_PIXELS = 1400 * 1400

# Assets directory
ASSETS_DIR = PROJECT_ROOT / "vision_tokenization" / "qualitative_benchmark" / "assets"

---
## 1. Image Reconstruction with Emu3.5 IBQ

Load the Emu3.5 IBQ vision tokenizer, encode an image to discrete tokens, then decode back and compare.

In [ ]:
from Tokenizer.Emu3_5_IBQ import Emu3_5_IBQ

vision_tokenizer = Emu3_5_IBQ(
    model_path=VISION_TOKENIZER_PATH,
    min_pixels=MIN_PIXELS,
    max_pixels=MAX_PIXELS,
    device=VISION_DEVICE,
)
print(f"Codebook size: {vision_tokenizer.codebook_size}")
print(f"Spatial factor: {vision_tokenizer.spatial_factor}x downsampling")

In [ ]:
# Pick an image from the benchmark assets (or use your own path)
image_path = ASSETS_DIR / "cute_dog.jpg"  # <-- change to any image

original = Image.open(image_path).convert("RGB")
print(f"Original size: {original.size} ({original.width * original.height:,} pixels)")

# Preprocess (smart resize + normalize)
img_tensor = vision_tokenizer.preprocess(original)
print(f"Preprocessed tensor: {img_tensor.shape}")

# Encode to discrete tokens
with torch.no_grad():
    indices, additional_info = vision_tokenizer.encode(img_tensor)

num_tokens = vision_tokenizer.get_num_tokens(indices)
h, w = indices.shape[-2], indices.shape[-1]
print(f"Token grid: {h} x {w} = {num_tokens:,} tokens")
print(f"Compression ratio: {original.width * original.height / num_tokens:.1f}x")

# Decode back to image
with torch.no_grad():
    reconstructed_tensor = vision_tokenizer.decode(indices, additional_info)

reconstructed = vision_tokenizer.postprocess(reconstructed_tensor)
print(f"Reconstructed size: {reconstructed.size}")

In [ ]:
# Visualize original vs reconstruction
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(original)
axes[0].set_title(f"Original\n{original.size[0]}x{original.size[1]}")
axes[0].axis("off")

axes[1].imshow(reconstructed)
axes[1].set_title(f"Reconstructed\n{h}x{w} tokens ({num_tokens:,} total)")
axes[1].axis("off")

# Token heatmap
token_grid = indices[0].cpu().numpy() if indices.ndim == 3 else indices.cpu().numpy()
im = axes[2].imshow(token_grid, cmap="viridis", aspect="auto")
axes[2].set_title("Token Indices")
axes[2].set_xlabel("Width")
axes[2].set_ylabel("Height")
plt.colorbar(im, ax=axes[2], fraction=0.046)

plt.suptitle(f"Emu3.5 IBQ Reconstruction — {pathlib.Path(image_path).name}", fontsize=14)
plt.tight_layout()
plt.show()

### Batch reconstruction of multiple images

In [ ]:
# Reconstruct several benchmark images
sample_images = [
    "cute_dog.jpg",
    "building.jpg",
    "line_chart.png",
    "math_draft1.png",
    "logo1.png",
    "checkerboard_pattern.png",
]

fig, axes = plt.subplots(2, len(sample_images), figsize=(4 * len(sample_images), 9))

for col, name in enumerate(sample_images):
    img = Image.open(ASSETS_DIR / name).convert("RGB")
    tensor = vision_tokenizer.preprocess(img)

    with torch.no_grad():
        idx, info = vision_tokenizer.encode(tensor)
        recon_tensor = vision_tokenizer.decode(idx, info)

    recon = vision_tokenizer.postprocess(recon_tensor)
    n_tok = vision_tokenizer.get_num_tokens(idx)

    axes[0, col].imshow(img)
    axes[0, col].set_title(name, fontsize=9)
    axes[0, col].axis("off")

    axes[1, col].imshow(recon)
    axes[1, col].set_title(f"{idx.shape[-2]}x{idx.shape[-1]} = {n_tok} tok", fontsize=9)
    axes[1, col].axis("off")

axes[0, 0].set_ylabel("Original", fontsize=12)
axes[1, 0].set_ylabel("Reconstructed", fontsize=12)
plt.suptitle("Emu3.5 IBQ — Reconstruction Gallery", fontsize=14)
plt.tight_layout()
plt.show()

---
## 2. Image Captioning with Apertus

Load the Apertus VLM and generate captions by encoding an image into vision tokens
and feeding them through the language model with a captioning prompt.

> **Note:** The current configuration assumes an **instruct/SFT model checkpoint**.
> The chat template (`apply_chat_template=True`, `chat_transform="to_apertus"`) wraps
> prompts in the Apertus instruct format (system + user turns).
>
> **For a base (non-instruct) checkpoint**, change the `InferenceArgs` to:
> ```python
> inf_args = InferenceArgs(
>     apply_chat_template=False,   # no chat template
>     chat_transform=None,         # not needed
>     ...
> )
> ```
> This will use simple image-tokens + text concatenation instead of the chat template.

In [ ]:
from vision_tokenization.qualitative_benchmark.v_tokenizers import create_vision_tokenizer
from vision_tokenization.qualitative_benchmark.inferencers.hf_inferencer import HFInferencer
from vision_tokenization.qualitative_benchmark.vlm import VLM, InferenceArgs

# Create the VLM vision tokenizer wrapper (reuses the Emu3.5 IBQ tokenizer for VLM formatting)
vlm_vision_tokenizer = create_vision_tokenizer(
    "emu3.5-ibq",
    model_path=VISION_TOKENIZER_PATH,
    min_pixels=MIN_PIXELS,
    max_pixels=MAX_PIXELS,
    device=VISION_DEVICE,
    tokenizer_path=TEXT_TOKENIZER_PATH,
)
print(f"VLM vision tokenizer: {vlm_vision_tokenizer.name}")

In [ ]:
# Load the Apertus language model via HuggingFace backend
inferencer = HFInferencer(
    model_path=APERTUS_MODEL_PATH,
    tokenizer_path=TEXT_TOKENIZER_PATH,
    device=MODEL_DEVICE,
    model_dtype="auto",
    max_seq_len=8192,
)
print("Apertus model loaded.")

In [ ]:
# Create VLM wrapper that ties together vision tokenizer + inferencer
inf_args = InferenceArgs(
    apply_chat_template=True,
    chat_transform="to_apertus",  # use "to_llama" for LLaMA-based chat template
    temperature=0.3,
    top_p=0.95,
    stop_token_ids=[],  # auto-populated from tokenizer config (sft_eot_token) during VLM init
    max_new_tokens=512,
    max_emu_aspect_ratio=MAX_PIXELS,
    min_emu_aspect_ratio=MIN_PIXELS,
)

vlm = VLM(
    vision_tokenizer=vlm_vision_tokenizer,
    inferencer=inferencer,
    inf_args=inf_args,
    tokenizer_path=TEXT_TOKENIZER_PATH,
    model_path=APERTUS_MODEL_PATH,
)
print("VLM ready for inference.")
print(f"Stop token IDs: {inf_args.stop_token_ids}")

In [ ]:
# Caption an image
caption_image_path = str(ASSETS_DIR / "cute_dog.jpg")  # <-- change to any image
caption_prompt = "Describe this image in detail."

print(f"Image: {caption_image_path}")
print(f"Prompt: {caption_prompt}")
print("Generating caption...")

formatted_prompt = vlm.preprocess(caption_image_path, caption_prompt)
caption = vlm.generate(formatted_prompt)

print("\n" + "=" * 60)
print("CAPTION:")
print("=" * 60)
print(caption)
print("=" * 60)

In [ ]:
# Visualize the image with its caption
import textwrap

img = Image.open(caption_image_path).convert("RGB")

fig, ax = plt.subplots(1, 1, figsize=(8, 8))
ax.imshow(img)
ax.axis("off")
ax.set_title(pathlib.Path(caption_image_path).name, fontsize=12)

wrapped = textwrap.fill(caption, width=90)
fig.text(
    0.5, 0.02, wrapped, ha="center", fontsize=11, wrap=True,
    bbox=dict(boxstyle="round,pad=0.5", facecolor="lightyellow", alpha=0.9),
)
plt.tight_layout()
plt.subplots_adjust(bottom=0.15)
plt.show()

### Caption multiple benchmark images

In [ ]:
caption_images = [
    "building.jpg",
    "line_chart.png",
    "3objects.png",
]
caption_prompt = "Describe this image briefly."

results = []
for name in caption_images:
    img_path = str(ASSETS_DIR / name)
    formatted = vlm.preprocess(img_path, caption_prompt)
    text = vlm.generate(formatted)
    results.append((name, text))
    print(f"[{name}] {text[:120]}..." if len(text) > 120 else f"[{name}] {text}")

# Display grid
fig, axes = plt.subplots(1, len(caption_images), figsize=(6 * len(caption_images), 7))
if len(caption_images) == 1:
    axes = [axes]

for ax, (name, text) in zip(axes, results):
    img = Image.open(ASSETS_DIR / name).convert("RGB")
    ax.imshow(img)
    ax.axis("off")
    wrapped = textwrap.fill(text, width=40)
    ax.set_title(f"{name}\n\n{wrapped}", fontsize=9)

plt.suptitle("Apertus Image Captioning", fontsize=14)
plt.tight_layout()
plt.show()

---
## 3. Visual Question Answering

Provide an image and a free-form question, and get a response from Apertus.

> This section reuses the VLM from Section 2. The same instruct-model settings
> apply — see the note there if you are using a base checkpoint instead.

In [ ]:
# ── Configure your image and question here ──

vqa_image_path = str(ASSETS_DIR / "immo_ex.png")  # <-- change to any image
vqa_prompt = "What does this image show?"  # <-- change to any question

print(f"Image: {vqa_image_path}")
print(f"Prompt: {vqa_prompt}")
print("Generating response...")

formatted_prompt = vlm.preprocess(vqa_image_path, vqa_prompt)
response = vlm.generate(formatted_prompt)

print("\n" + "=" * 60)
print("RESPONSE:")
print("=" * 60)
print(response)
print("=" * 60)

In [ ]:
# Visualize
img = Image.open(vqa_image_path).convert("RGB")

fig, ax = plt.subplots(1, 1, figsize=(8, 8))
ax.imshow(img)
ax.axis("off")
ax.set_title(pathlib.Path(vqa_image_path).name, fontsize=12)

prompt_text = textwrap.fill(f"Q: {vqa_prompt}", width=90)
answer_text = textwrap.fill(f"A: {response}", width=90)

fig.text(
    0.5, 0.12, prompt_text, ha="center", fontsize=11,
    bbox=dict(boxstyle="round,pad=0.4", facecolor="lightblue", alpha=0.9),
)
fig.text(
    0.5, 0.02, answer_text, ha="center", fontsize=11,
    bbox=dict(boxstyle="round,pad=0.4", facecolor="lightyellow", alpha=0.9),
)
plt.tight_layout()
plt.subplots_adjust(bottom=0.22)
plt.show()

### Try different prompts on the same image

In [ ]:
vqa_image_path = str(ASSETS_DIR / "math_draft1.png")  # <-- change to any image

prompts = [
    "What mathematical equations are shown in this image?",
    "Transcribe all the text in this image.",
    "What topic of mathematics is this about?",
]

print(f"Image: {vqa_image_path}\n")
for prompt in prompts:
    formatted = vlm.preprocess(vqa_image_path, prompt)
    resp = vlm.generate(formatted)
    print(f"Q: {prompt}")
    print(f"A: {resp}\n")

### Use your own image

In [ ]:
# Point to any image on disk
custom_image_path = "/path/to/your/image.jpg"  # <-- EDIT THIS
custom_prompt = "What do you see in this image?"  # <-- EDIT THIS

if os.path.exists(custom_image_path):
    formatted = vlm.preprocess(custom_image_path, custom_prompt)
    resp = vlm.generate(formatted)

    img = Image.open(custom_image_path).convert("RGB")
    fig, ax = plt.subplots(1, 1, figsize=(8, 8))
    ax.imshow(img)
    ax.axis("off")
    ax.set_title(f"Q: {custom_prompt}")
    fig.text(
        0.5, 0.02, textwrap.fill(f"A: {resp}", width=90),
        ha="center", fontsize=11,
        bbox=dict(boxstyle="round,pad=0.4", facecolor="lightyellow", alpha=0.9),
    )
    plt.tight_layout()
    plt.subplots_adjust(bottom=0.12)
    plt.show()
else:
    print(f"Image not found: {custom_image_path}")
    print("Edit custom_image_path above to point to a real image.")

---
## Cleanup

In [ ]:
# Free GPU memory
del vlm, inferencer, vlm_vision_tokenizer, vision_tokenizer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    for i in range(torch.cuda.device_count()):
        free, total = torch.cuda.mem_get_info(i)
        print(f"GPU {i}: {free/1024**3:.1f}GB free / {total/1024**3:.1f}GB total")